# [재실행 전용 · 2026-10-04 회계 수정] crypto/notebooks/crypto_pit_walkforward_colab.ipynb

원본 `crypto/notebooks/crypto_pit_walkforward_colab.ipynb`에서 **§A(백본 재학습)와 §B 풀 학습 셀을 빼고, 저장된 4β 풀로 평가만 다시 하는 노트북이다**. 환경은 회계 수정판(`_drift_weights` 등)이 들어간 원본 셀 그대로다.
위에서부터 **순서대로 전부 실행**하면 된다. 결과는 원본과 같은 이름으로 저장되고, 실행 전 백업 셀이 수정 전 결과를 `_pre_accounting_fix_20261004/`로 먼저 복사한다(이미 복사된 파일은 건너뜀).

결과: 7장 point-in-time 절(검증 스윕 B-8, 테스트 1회 B-9, 일관성 표 B-10, 롤링 fold C-2, 다양성 C-3).


In [ ]:
# 수정 전 결과 백업 — 파일 단위로, 이미 백업된 파일은 덮어쓰지 않음
import shutil, glob, os
from google.colab import drive
drive.mount('/content/drive')
src = '/content/drive/MyDrive/졸업프로젝트/data/crypto'
dst = os.path.join(src, '_pre_accounting_fix_20261004')
os.makedirs(dst, exist_ok=True)
copied = []
for pat in ['crypto_pit_*.csv', 'crypto_pit_*.png']:
    for p in glob.glob(os.path.join(src, pat)):
        target = os.path.join(dst, os.path.basename(p))
        if not os.path.exists(target):
            shutil.copy2(p, target)
            copied.append(os.path.basename(p))
print('백업 폴더:', dst)
print('이번에 백업한 파일:', copied if copied else '없음(이미 백업됨)')


## §B. RL 최종 레시피 재현 (Stage A/B + valid 기반 β·α 선택, 라우터 없음)

라우터(Stage C)는 배포 레시피에서 이미 기여 0으로 확인됐으므로(원 논문 4.5–4.6절) 여기서는 처음부터 제외한다. top-K-플러스-게이트(v4)도 13종목이라는 작은 행동공간에서는 구조적 필요성이 약해 생략하고, v1 스타일 밀집 softmax + β-조건부 청크 샘플링 + 추론 시점 스무딩만 이식한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
import warnings
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
CRYPTO_DIR = DATA_DIR / 'crypto'
FEAT_DIR   = CRYPTO_DIR / 'features_5m'
MODEL_DIR  = DRIVE_ROOT / 'models'
POOL_DIR   = MODEL_DIR / 'crypto_hrl_pool'

RL_EMB_H5      = CRYPTO_DIR / 'crypto_pit_rl_embeddings.h5'
POOL_DIR       = MODEL_DIR / 'crypto_pit_hrl_pool'   # §2 상수와 별도 디렉토리(44종목 풀과 섞이지 않도록)

EMB_DIM    = 64
TC         = 0.001    # 거래비용 (Binance taker fee 기준)
BUCKET_SEC = 1800     # 30분 버킷

RL_TRAIN_START = '2021-01-01'
RL_TRAIN_END   = '2023-12-31'
RL_VALID_START = '2024-01-01'
RL_VALID_END   = '2024-12-31'
RL_TEST_START  = '2025-01-01'
RL_TEST_END    = '2099-12-31'   # 열린 구간 — 실제로는 데이터 끝(~2026-07)까지

# --- Stage A/B/C 하이퍼파라미터 ---
N_LABELS              = 5          # 레짐 라벨 수 (bear/pullback/sideways/rally/bull)
RISK_THRESHOLD        = 0.2        # EarnHFT θ — 상하위 라벨 경계 분위수 폭
LOWPASS_WINDOW        = 48         # 레짐 분할 저역통과 윈도우 = 1일(30분×48)
POOL_BETAS            = [-90, -30, 30, 90]   # Stage B 선호도 스윕 (논문 [-90,-10,30,100] 근사)
POOL_TIMESTEPS_EACH   = 100_000    # 풀 에이전트 1개당 학습 스텝 (원본 300k와 총량 비슷하게)
SMOOTH_ALPHAS          = [1.0, 0.3, 0.15, 0.05, 0.03, 0.02, 0.01]  # 라우터 없음 → §9 스타일 스윕만

assert RL_EMB_H5.exists(), 'crypto_pit_rl_embeddings.h5 없음 — 위 §A를 먼저 실행'
print('경로 확인 완료')


## 1. RL 임베딩 로드 & 인덱스 구축

심볼마다 5분봉 backfill 시작 시각이 달라 30분 간격 샘플의 초 단위 timestamp가 서로 어긋난다(위상 차이). `ts // BUCKET_SEC`로 30분 버킷에 매핑하면 심볼별로 정확히 버킷당 1개씩 떨어져(STRIDE=6이 정확히 1버킷과 같으므로 충돌 없음) 서로 다른 위상이라도 같은 30분 창으로 정렬된다.

In [ ]:
print('crypto_rl_embeddings.h5 로드 중...')
with h5py.File(RL_EMB_H5, 'r') as f:
    symbols_raw = f['symbols'][:]
    ts_raw      = f['timestamps'][:]     # epoch seconds (UTC)
    embs_all    = f['embeddings'][:]     # (N, 64)

symbols_list = [s.decode() for s in symbols_raw]
ts_list      = ts_raw.astype(np.int64).tolist()

# (symbol, 30분 버킷) → 임베딩 인덱스
emb_index  = {}
sym_ts_set = {}   # 심볼별로 임베딩 추출에 실제 쓰인 초 단위 timestamp 집합 (수익률 정확 매칭용)
for i, (s, t) in enumerate(zip(symbols_list, ts_list)):
    emb_index[(s, t // BUCKET_SEC)] = i
    sym_ts_set.setdefault(s, set()).add(t)

symbols  = sorted(set(symbols_list))
N_ASSETS = len(symbols)
print(f'임베딩: {embs_all.shape} | 심볼: {N_ASSETS}개 | 인덱스: {len(emb_index):,}개')


## 2. 30분 수익률(RetTarget_6b) 로드

In [ ]:
print('features_5m parquet에서 RetTarget_6b 로드 중...')
files = sorted(glob.glob(str(FEAT_DIR / '*.parquet')))
assert files, f'피처 parquet 없음: {FEAT_DIR}'

rl_start_ts = pd.Timestamp(RL_TRAIN_START, tz='UTC')  # = 2021-01-01, §A와 동일 시작점
ret_index   = {}

for fp in files:
    sym = Path(fp).stem
    valid_ts = sym_ts_set.get(sym)
    if not valid_ts:
        continue
    df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])
    df = df[df['datetime'] >= rl_start_ts]
    epoch = df['datetime'].values.astype('int64') // 10**9   # ts_list와 동일 단위(초)
    y     = df['RetTarget_6b'].values
    # 임베딩 추출에 실제 쓰인 timestamp만 매칭 — 같은 버킷 내 다른 5분봉과 섞이지 않도록
    for t, r in zip(epoch, y):
        t = int(t)
        if t in valid_ts and not np.isnan(r):
            ret_index[(sym, t // BUCKET_SEC)] = float(r)

all_buckets = sorted(set(t // BUCKET_SEC for t in ts_list))
print(f'수익률 인덱스: {len(ret_index):,}개 | 전체 30분 버킷: {len(all_buckets):,}개')


## 3. CryptoPortfolioEnv (Gymnasium) — 기존과 동일, 변경 없음

In [ ]:
# [2026-10-04 회계 수정] 가격 변동 후 실제 보유 비중(drift)·거래 가능 종목 마스크.
# 이전 버전은 수익 반영 뒤에도 직전 '목표' 비중을 보유 비중으로 저장해, 가격 변동으로 틀어진 비중을
# 되돌리는 재조정 거래(등가중 포함)의 비용이 0으로 계산됐다.
def _drift_weights(w, rets):
    """수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr). 합<1이면 나머지는 현금(수익률 0)으로 남는다."""
    w = np.asarray(w, dtype=np.float64)
    g = 1.0 + float(np.dot(w, rets))
    if g <= 1e-12:
        return w.astype(np.float32)
    return (w * (1.0 + np.asarray(rets, dtype=np.float64)) / g).astype(np.float32)


def _mask_untradable(w, tradable, renorm=True):
    """수익률 데이터가 없는(상장 전·상폐 후) 종목 비중을 0으로. renorm=False면 빠진 몫은 현금으로 남는다."""
    w = np.where(tradable, np.asarray(w, dtype=np.float64), 0.0)
    s = w.sum()
    return (w / s if renorm and s > 1e-12 else w).astype(np.float32)


class CryptoPortfolioEnv(gym.Env):
    """
    크립토 44종목 개별 자산 기반 포트폴리오 운용 환경 (클러스터링 없음)

    State:  종목별 임베딩(N×64) + 현재 비중(N) → N*64+N dim
    Action: 종목별 목표 포트폴리오 비중 (N-dim, Softmax 정규화)
    Reward: 30분 포트폴리오 수익률(RetTarget_6b, 이미 forward-looking) − 거래비용
    스텝 = 30분 버킷(BUCKET_SEC) 단위 — emb_index/ret_index 모두 (symbol, bucket) 키.
    """
    metadata = {'render_modes': []}

    def __init__(self, buckets, symbols, emb_index, embs_all, ret_index,
                 emb_dim=64, tc=0.001):
        super().__init__()
        self.buckets    = buckets
        self.symbols    = symbols
        self.n_assets   = len(symbols)
        self.emb_index  = emb_index
        self.embs_all   = embs_all
        self.ret_index  = ret_index
        self.emb_dim    = emb_dim
        self.tc         = tc

        obs_dim = self.n_assets * emb_dim + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets,), dtype=np.float32)

    def _asset_emb(self, bucket):
        """종목별 임베딩 — 해당 버킷에 데이터 없으면 0-vector(상장 전/상폐 등)"""
        embs = []
        for sym in self.symbols:
            idx = self.emb_index.get((sym, bucket))
            embs.append(self.embs_all[idx] if idx is not None else np.zeros(self.emb_dim, dtype=np.float32))
        return np.concatenate(embs)  # (N*64,)

    def _asset_ret(self, bucket):
        """종목별 30분 수익률 — 없으면 0(그 자산은 보유해도 손익 없음으로 취급)"""
        return np.array([self.ret_index.get((sym, bucket), 0.0) for sym in self.symbols], dtype=np.float32)

    def _tradable(self, bucket):
        """해당 버킷에 수익률 데이터가 있는(거래 가능한) 종목 마스크"""
        return np.array([(sym, bucket) in self.ret_index for sym in self.symbols])

    def _get_obs(self):
        b = self.buckets[self.t]
        return np.concatenate([self._asset_emb(b), self.weights]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t               = 0
        self.weights         = _mask_untradable(np.ones(self.n_assets, dtype=np.float32),
                                          self._tradable(self.buckets[0]))  # 거래 가능 종목 등가중에서 시작
        self.portfolio_value = 1.0
        self.value_history   = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        # Softmax로 비중 정규화
        action  = np.exp(action - action.max())
        action  = action / action.sum()
        action  = _mask_untradable(action, self._tradable(self.buckets[self.t]))  # 거래 불가 종목 제외

        # 거래비용
        turnover = float(np.abs(action - self.weights).sum())  # 비중 변화량(진단용)
        tc_cost  = self.tc * turnover

        # RetTarget_6b는 현재 버킷 자체에 forward 수익률이 이미 인코딩되어 있음
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((action * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(action, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)

        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover, 'tc_cost': tc_cost}
        return self._get_obs(), reward, done, False, info


In [ ]:
def bucket_to_ts(b):
    return pd.Timestamp(b * BUCKET_SEC, unit='s', tz='UTC')

def bucket_to_date(b):
    return bucket_to_ts(b).strftime('%Y-%m-%d')

train_buckets = [b for b in all_buckets if RL_TRAIN_START <= bucket_to_date(b) <= RL_TRAIN_END]
valid_buckets = [b for b in all_buckets if RL_VALID_START <= bucket_to_date(b) <= RL_VALID_END]
test_buckets  = [b for b in all_buckets if RL_TEST_START  <= bucket_to_date(b) <= RL_TEST_END]
print(f'학습 기간: {bucket_to_ts(train_buckets[0])} ~ {bucket_to_ts(train_buckets[-1])} ({len(train_buckets):,}스텝)')
print(f'검증 기간: {bucket_to_ts(valid_buckets[0])} ~ {bucket_to_ts(valid_buckets[-1])} ({len(valid_buckets):,}스텝)')
print(f'테스트 기간: {bucket_to_ts(test_buckets[0])} ~ {bucket_to_ts(test_buckets[-1])} ({len(test_buckets):,}스텝)')

env_kwargs = dict(
    symbols=symbols, emb_index=emb_index, embs_all=embs_all,
    ret_index=ret_index, emb_dim=EMB_DIM, tc=TC,
)

print('\n환경 유효성 검사...')
check_env(CryptoPortfolioEnv(train_buckets[:100], **env_kwargs), warn=True)
print('OK')

## 4. Stage A — 시장 레짐 분할 (EarnHFT Algorithm 3)

44종목 등가중 지수(equal-weight NAV curve)를 만들어 저역통과 필터 → 극값 분할 → 인접구간 병합(기울기차+DTW) →
기울기 분위수 라벨링을 적용한다. 결과는 (a) 각 30분 버킷의 레짐 라벨, (b) 라벨이 바뀌지 않는 연속 구간(chunk) 목록.

로컬 `crypto/market_segmentation.py`와 동일한 코드(Colab은 로컬 저장소를 마운트하지 않으므로 셀에 직접 포함).
로컬에서 BTC 종가로 검증했을 때(1일 저역통과, 2025-01~2026-07 구간) 936개 극값 기반 구간이 187개 레짐 구간(평균 약
3일)으로 병합되고 sideways/rally/pullback/bear/bull 5개 라벨에 고르게 분포하는 것을 확인했다.

In [ ]:
# crypto/market_segmentation.py 이식 (Colab은 저장소를 마운트하지 않으므로 동일 코드를 셀에 포함)
REGIME_NAMES = {1: 'bear', 2: 'pullback', 3: 'sideways', 4: 'rally', 5: 'bull'}
_DTW_MAX_LEN = 200


@dataclass
class MarketSegments:
    labels: pd.Series
    chunks: pd.DataFrame


def _zscore(x):
    std = x.std()
    return (x - x.mean()) / std if std > 1e-12 else np.zeros_like(x)


def _downsample(x, max_len=_DTW_MAX_LEN):
    if len(x) <= max_len:
        return x
    idx = np.linspace(0, len(x) - 1, max_len).round().astype(int)
    return x[idx]


def _dtw_distance(a, b):
    a, b = _downsample(a), _downsample(b)
    n, m = len(a), len(b)
    cost = np.full((n + 1, m + 1), np.inf)
    cost[0, 0] = 0.0
    for i in range(1, n + 1):
        ai = a[i - 1]
        row, prev = cost[i], cost[i - 1]
        for j in range(1, m + 1):
            row[j] = abs(ai - b[j - 1]) + min(prev[j], row[j - 1], prev[j - 1])
    return float(cost[n, m])


def _slope(x):
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x), dtype=np.float64)
    a, _ = np.polyfit(t, x, 1)
    return float(a)


def _find_extrema_indices(x):
    d = np.diff(x)
    sign = np.sign(d)
    sign[sign == 0] = 1
    change = np.where(np.diff(sign) != 0)[0] + 1
    idx = np.concatenate(([0], change, [len(x) - 1]))
    return np.unique(idx)


def _merge_pass(x, bounds, merge_quantile):
    if len(bounds) <= 2:
        return bounds, False
    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
    if len(seg_ranges) < 2:
        return bounds, False
    slope_diffs = [abs(slopes[i + 1] - slopes[i]) for i in range(len(slopes) - 1)]
    dtw_dists = [
        _dtw_distance(
            _zscore(x[seg_ranges[i][0]:seg_ranges[i][1] + 1]),
            _zscore(x[seg_ranges[i + 1][0]:seg_ranges[i + 1][1] + 1]),
        )
        for i in range(len(seg_ranges) - 1)
    ]
    slope_thresh = np.quantile(slope_diffs, merge_quantile)
    dtw_thresh = np.quantile(dtw_dists, merge_quantile)

    new_bounds = [bounds[0]]
    merged_any = False
    i = 0
    while i < len(seg_ranges):
        can_merge = (
            i < len(seg_ranges) - 1
            and slope_diffs[i] <= slope_thresh
            and dtw_dists[i] <= dtw_thresh
        )
        if can_merge:
            new_bounds.append(seg_ranges[i + 1][1])
            merged_any = True
            i += 2
        else:
            new_bounds.append(seg_ranges[i][1])
            i += 1
    return new_bounds, merged_any


def _absorb_short_segments(x, bounds, min_length):
    while len(bounds) > 2:
        seg_ranges = list(zip(bounds[:-1], bounds[1:]))
        lengths = [b - a + 1 for a, b in seg_ranges]
        shortest = int(np.argmin(lengths))
        if lengths[shortest] >= min_length:
            break
        slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
        left_diff = abs(slopes[shortest] - slopes[shortest - 1]) if shortest > 0 else np.inf
        right_diff = (
            abs(slopes[shortest] - slopes[shortest + 1]) if shortest < len(seg_ranges) - 1 else np.inf
        )
        drop_idx = shortest if left_diff <= right_diff else shortest + 1
        bounds = bounds[:drop_idx] + bounds[drop_idx + 1:]
    return bounds


def segment_and_label(series, n_labels=5, risk_threshold=0.2, lowpass_window=48,
                       merge_quantile=0.35, max_merge_iters=50, min_segment_length=None):
    """가격 시계열을 레짐 구간으로 분할하고 라벨을 부여한다 (EarnHFT Algorithm 3 이식).
    로그가격(buy&hold net curve의 로그) 기준으로 계산해 가격 스케일 왜곡을 방지한다."""
    x_raw = series.to_numpy(dtype=np.float64)
    if len(x_raw) < 3:
        raise ValueError('series가 너무 짧습니다 (최소 3개 포인트 필요)')
    if np.any(x_raw <= 0):
        raise ValueError('series는 로그 변환을 위해 모두 양수여야 합니다')

    x_log = np.log(x_raw)
    x_smooth = pd.Series(x_log).rolling(lowpass_window, min_periods=1, center=True).mean().to_numpy()

    bounds = _find_extrema_indices(x_smooth).tolist()
    for _ in range(max_merge_iters):
        bounds, merged = _merge_pass(x_smooth, bounds, merge_quantile)
        if not merged or len(bounds) <= 2:
            break
    bounds = _absorb_short_segments(x_log, bounds, min_segment_length or lowpass_window)

    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = np.array([_slope(x_log[a:b + 1]) for a, b in seg_ranges])

    theta = risk_threshold
    H = np.quantile(slopes, 1 - theta / 2)
    L = np.quantile(slopes, theta / 2)

    labels = np.empty(len(slopes), dtype=np.int64)
    for i, s in enumerate(slopes):
        if s > H:
            labels[i] = n_labels
        elif s < L:
            labels[i] = 1
        elif H > L:
            frac = (s - L) / (H - L)
            labels[i] = int(np.clip(2 + frac * (n_labels - 2), 2, n_labels - 1))
        else:
            labels[i] = (n_labels + 1) // 2

    per_point_labels = np.empty(len(x_raw), dtype=np.int64)
    for (a, b), lbl in zip(seg_ranges, labels):
        per_point_labels[a:b + 1] = lbl

    labels_series = pd.Series(per_point_labels, index=series.index, name='regime_label')

    chunks = pd.DataFrame({
        'start_idx': [a for a, _ in seg_ranges],
        'end_idx': [b for _, b in seg_ranges],
        'length': [b - a + 1 for a, b in seg_ranges],
        'slope': slopes,
        'label': labels,
    })
    chunks['regime'] = chunks['label'].map(REGIME_NAMES) if n_labels == 5 else chunks['label'].astype(str)
    return MarketSegments(labels=labels_series, chunks=chunks)


def chunk_sampling_priority(chunks, beta, risk_threshold=0.2):
    """EarnHFT Eq.4 이식 — chunk 기울기(r)에 대한 β-지수가중 샘플링 우선순위(합 1)."""
    r = chunks['slope'].to_numpy(dtype=np.float64)
    theta = risk_threshold
    Ht = np.quantile(r, 1 - theta / 2)
    Lt = np.quantile(r, theta / 2)

    std = r.std() if r.std() > 1e-12 else 1.0
    h = max(0.9 * std * len(r) ** (-1 / 5), 1e-8)
    diffs = (r[:, None] - r[None, :]) / h
    pdf = np.exp(-0.5 * diffs ** 2).sum(axis=1) / (len(r) * h * np.sqrt(2 * np.pi))
    pdf = np.maximum(pdf, 1e-12)

    tail_mask = (r >= Ht) | (r <= Lt)
    log_w = beta * r - np.where(tail_mask, 0.0, np.log(pdf))
    log_w -= log_w.max()
    weights = np.exp(log_w)
    return weights / weights.sum()


print('market_segmentation 함수 정의 완료')


In [ ]:
# 44종목 등가중 NAV 곡선 구성 (거래비용 제외 — 시장 자체의 추세 파악용)
eq_ret_per_bucket = np.array([
    np.mean([ret_index.get((sym, b), 0.0) for sym in symbols])
    for b in train_buckets
])
eq_nav = pd.Series(np.cumprod(1 + eq_ret_per_bucket), index=range(len(train_buckets)))

seg = segment_and_label(
    eq_nav, n_labels=N_LABELS, risk_threshold=RISK_THRESHOLD,
    lowpass_window=LOWPASS_WINDOW,
)
print(f"구간 수: {len(seg.chunks)} (평균 길이 {seg.chunks['length'].mean():.0f}버킷 ≈ {seg.chunks['length'].mean()/48:.1f}일)")
print(seg.chunks['regime'].value_counts())

fig, ax = plt.subplots(figsize=(14, 4))
colors = {'bear': 'firebrick', 'pullback': 'lightsalmon', 'sideways': 'lightgray', 'rally': 'lightgreen', 'bull': 'darkgreen'}
for _, row in seg.chunks.iterrows():
    ax.axvspan(row['start_idx'], row['end_idx'], color=colors[row['regime']], alpha=0.5)
ax.plot(eq_nav.values, color='black', lw=1)
ax.set_title('44종목 등가중 NAV + Stage A 레짐 분할 (2025 학습 구간, 2021–2023)')
ax.set_xlabel('버킷 인덱스 (train_buckets 내 위치)')
plt.tight_layout()
plt.show()


## 5. Stage B — 레짐 선호도 기반 에이전트 풀 학습

`CryptoPortfolioEnv`를 "청크 단위 에피소드"로 감싼 `ChunkedCryptoPortfolioEnv`를 만든다. `reset()`마다
EarnHFT Eq.4 우선순위(β)로 Stage A의 chunk 하나를 뽑아 그 구간만 진행하고 종료한다. step()/관측/보상
로직은 `CryptoPortfolioEnv`와 완전히 동일 — 에피소드 경계와 시작 구간만 다르다.

`POOL_BETAS`의 각 β로 별도 PPO를 학습해 4개 에이전트 풀을 만든다(β>0: 강세장 선호 구간 위주 학습,
β<0: 약세장 선호 구간 위주 학습).

In [ ]:
class ChunkedCryptoPortfolioEnv(CryptoPortfolioEnv):
    """
    EarnHFT Stage II 이식 — reset()마다 레짐 우선순위(β)로 청크를 샘플링해
    그 구간만 진행하고 종료하는 에피소드형 환경.
    """

    def __init__(self, all_buckets, chunks, beta, risk_threshold=0.2, rng=None, **env_kwargs):
        self.all_buckets = all_buckets
        self.chunks      = chunks.reset_index(drop=True)
        self.beta        = beta
        self.priority    = chunk_sampling_priority(self.chunks, beta, risk_threshold)
        self.rng         = rng or np.random.default_rng()
        super().__init__(buckets=all_buckets[:2], **env_kwargs)  # reset()에서 실제 구간으로 교체됨

    def reset(self, seed=None, options=None):
        idx = self.rng.choice(len(self.chunks), p=self.priority)
        row = self.chunks.iloc[idx]
        self.buckets = self.all_buckets[int(row['start_idx']):int(row['end_idx']) + 1]
        return super().reset(seed=seed, options=options)


# 스모크 테스트: β별로 몇 번 reset해서 실제로 다른 구간이 뽑히는지 확인
_test_env = ChunkedCryptoPortfolioEnv(
    all_buckets=train_buckets, chunks=seg.chunks, beta=90, rng=np.random.default_rng(0), **env_kwargs
)
for _ in range(3):
    _test_env.reset()
    print(f'β=90 샘플 구간: {len(_test_env.buckets)}버킷 ({bucket_to_date(_test_env.buckets[0])} ~ {bucket_to_date(_test_env.buckets[-1])})')


In [ ]:
# 학습 셀 대신 저장된 4β 풀 로드 (2026-08-18 학습본, models/crypto_pit_hrl_pool/)
if 'pool_models' not in globals() or not pool_models:
    pool_models = {b: PPO.load(str(POOL_DIR / f'agent_beta_{b}.zip')) for b in POOL_BETAS}
    print('4β 풀 로드 완료:', list(pool_models.keys()))


### B-6. 헬퍼 함수 (compute_perf / run_backtest / 단독 스무딩 실행)

In [ ]:
def compute_perf(values, steps_per_year=48 * 365):
    """성과 지표 계산 — 30분봉·24/7 시장 기준 연환산(48스텝/일)"""
    rets       = np.diff(values) / values[:-1]
    total_ret  = values[-1] / values[0] - 1
    annual_ret = (1 + total_ret) ** (steps_per_year / len(rets)) - 1
    sharpe     = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    max_dd     = ((values / np.maximum.accumulate(values)) - 1).min()
    return {'총 수익률': total_ret, '연환산 수익률': annual_ret, 'Sharpe': sharpe, 'MDD': max_dd}


def run_backtest(env, model, deterministic=True):
    obs, _ = env.reset()
    done, values = False, [1.0]
    while not done:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, _, done, _, info = env.step(action)
        values.append(info['portfolio_value'])
    return np.array(values), list(env.turnover_history)


def equal_weight_backtest(env):
    obs, _ = env.reset()
    done, values = False, [1.0]
    action = np.ones(env.n_assets, dtype=np.float32) / env.n_assets
    while not done:
        obs, _, done, _, info = env.step(action)
        values.append(info['portfolio_value'])
    return np.array(values), list(env.turnover_history)


def _softmax_np(a):
    e = np.exp(a - a.max())
    return e / e.sum()


def run_standalone_smoothed(model, buckets, alpha):
    """라우터 없이 단일 β 전문가에 추론 시점 관성 스무딩만 적용해 buckets 구간을 실행.
    반환: (비용 후 NAV, 비용 전 NAV, 턴오버 리스트) — §7 run_hrl_smoothed의 무라우터 버전."""
    env = CryptoPortfolioEnv(buckets, **env_kwargs)
    obs, _ = env.reset()
    gross, g, done = [1.0], 1.0, False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        w = (1 - alpha) * env.weights + alpha * _softmax_np(a)
        obs, _, done, _, info = env.step(np.log(w))
        g *= 1 + info['port_ret']
        gross.append(g)
    return np.array(env.value_history), np.array(gross), list(env.turnover_history)


print('헬퍼 함수 정의 완료')

### B-7. 검증구간(2024)에서 4β 원시 성과 확인

In [ ]:
valid_env = CryptoPortfolioEnv(valid_buckets, **env_kwargs)
test_env  = CryptoPortfolioEnv(test_buckets,  **env_kwargs)

raw_rows = {}
for beta in POOL_BETAS:
    v_values, _ = run_backtest(CryptoPortfolioEnv(valid_buckets, **env_kwargs), pool_models[beta])
    raw_rows[f'β={beta}'] = compute_perf(v_values)

ew_valid_values, _ = equal_weight_backtest(CryptoPortfolioEnv(valid_buckets, **env_kwargs))
raw_rows['등가중'] = compute_perf(ew_valid_values)

raw_df = pd.DataFrame(raw_rows).T
print('검증구간(2024) 원시(스무딩 없음) 성과 — β 선택 전 사전 확인용')
print(raw_df.to_string())

### B-8. 검증구간(2024)에서 β×α 스윕 — 테스트는 아직 보지 않음

4개 β × 7개 α = 28개 조합을 전부 **검증구간에서만** 평가해 최고 검증 샤프 조합을 고른다. 이 시점까지 테스트구간(2025-2026)은 코드 어디에서도 실행되지 않는다 — 원 논문 4.6/6.3절과 동일한 선택 편향 방지 절차.

In [ ]:
# 4개 β 전부에 대해 검증구간(2024)에서 α 스윕 → 검증 샤프가 가장 높은 (β, α) 조합을 선택.
# 원 논문 4.6절 방법론(테스트를 전혀 보지 않고 검증만으로 선택)을 그대로 따른다.
valid_sweep = {}
for beta in POOL_BETAS:
    model = pool_models[beta]
    for alpha in SMOOTH_ALPHAS:
        net, gross, turns = run_standalone_smoothed(model, valid_buckets, alpha)
        perf = compute_perf(net)
        valid_sweep[(beta, alpha)] = perf
    print(f'β={beta} valid 스윕 완료')

sweep_df = pd.DataFrame(valid_sweep).T
sweep_df.index.names = ['beta', 'alpha']
BEST_BETA, BEST_ALPHA = sweep_df['Sharpe'].idxmax()
print(f'\n검증 샤프 최고: β={BEST_BETA}, α={BEST_ALPHA} (검증 샤프 {sweep_df["Sharpe"].max():.3f})')
print()
print(sweep_df.sort_values('Sharpe', ascending=False).head(10).to_string())
sweep_df.to_csv(CRYPTO_DIR / 'crypto_pit_valid_sweep.csv', encoding='utf-8-sig')

### B-9. 최종 (β\*, α\*)를 테스트구간(2025-2026)에서 1회 평가

In [ ]:
print(f'[최종 검증] 검증구간에서 고른 (β={BEST_BETA}, α={BEST_ALPHA})로 테스트구간을 "1회만" 실행')

final_model = pool_models[BEST_BETA]
net, gross, turns = run_standalone_smoothed(final_model, test_buckets, BEST_ALPHA)
final_perf = compute_perf(net)
gross_perf = compute_perf(gross)

ew_test_values, ew_test_turns = equal_weight_backtest(CryptoPortfolioEnv(test_buckets, **env_kwargs))
ew_perf = compute_perf(ew_test_values)

final_df = pd.DataFrame({
    f'PIT 최종(β={BEST_BETA}, α={BEST_ALPHA}) — 비용 후': final_perf,
    f'PIT 최종 — 비용 전': gross_perf,
    '등가중 벤치마크(2025-2026 테스트)': ew_perf,
}).T
print()
print(final_df.to_string())
final_df.to_csv(CRYPTO_DIR / 'crypto_pit_final_test.csv', encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(net, label=f'PIT 최종(β={BEST_BETA}, α={BEST_ALPHA})', lw=1.8)
ax.plot(ew_test_values, label='등가중 벤치마크', lw=1.5, ls='--')
ax.axhline(1, color='gray', lw=0.8, ls=':')
ax.set_title(f'Point-in-Time 13종목 walk-forward — 테스트구간({RL_TEST_START}~) 누적가치')
ax.set_xlabel('30분 스텝')
ax.set_ylabel('포트폴리오 가치 (초기=1.0)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_pit_final_test.png', dpi=150, bbox_inches='tight')
plt.show()

### B-10. 4β 전체 검증/테스트 일관성 표

원 논문 4.6절 표와 같은 형식 — β=-30이 44종목 트랙에서처럼 여기서도 격차가 가장 작은지, 혹은 이 point-in-time 세팅에서는 다른 β가 더 견조한지를 직접 비교한다.

In [ ]:
# 원 논문 4.6절과 동일한 형식 — 4개 β 전부를 라우터 없이 단독으로, 각 β 자신의 검증-최고
# α로(검증 스윕에서 이미 계산됨) 검증구간·테스트구간 양쪽에서 실행해 |검증-테스트| 샤프
# 격차를 계산한다. 이 표가 §B-8에서 β=BEST_BETA를 고른 근거를 다른 β와 나란히 보여준다.
rows = []
for beta in POOL_BETAS:
    beta_sweep = {a: valid_sweep[(beta, a)]['Sharpe'] for a in SMOOTH_ALPHAS}
    best_alpha_for_beta = max(beta_sweep, key=beta_sweep.get)
    model = pool_models[beta]

    v_net, _, _ = run_standalone_smoothed(model, valid_buckets, best_alpha_for_beta)
    t_net, _, _ = run_standalone_smoothed(model, test_buckets,  best_alpha_for_beta)
    v_perf, t_perf = compute_perf(v_net), compute_perf(t_net)

    rows.append({
        'beta': beta, 'best_alpha(valid)': best_alpha_for_beta,
        'valid_sharpe': v_perf['Sharpe'], 'test_sharpe': t_perf['Sharpe'],
        'valid_ret': v_perf['총 수익률'], 'test_ret': t_perf['총 수익률'],
        'valid_test_gap': abs(v_perf['Sharpe'] - t_perf['Sharpe']),
    })

consistency_df = pd.DataFrame(rows).set_index('beta')
print('4β 전체 검증/테스트 일관성 표 (원 논문 4.6절 형식)')
print(consistency_df.to_string())
consistency_df.to_csv(CRYPTO_DIR / 'crypto_pit_consistency_table.csv', encoding='utf-8-sig')

### B-11. Point-in-Time 유니버스 감사 기록 (논문 인용용, 재실행 시 항상 저장)

이 셀은 백테스트와 무관하며, 유니버스 선정 근거를 논문에 그대로 인용할 수 있게 JSON으로 남긴다.

In [ ]:
# §A를 건너뛰므로 §A에서 정의되던 이름을 여기서 보충
import json
PIT_SYMBOLS = [  # point-in-time 유니버스 13종목 (§0 타이틀 셀 참고)
    'BTCUSDT', 'ETHUSDT', 'XRPUSDT', 'LTCUSDT', 'BCHUSDT', 'ADAUSDT', 'BNBUSDT',
    'LINKUSDT', 'XLMUSDT', 'TRXUSDT', 'DOTUSDT', 'THETAUSDT', 'XTZUSDT',
]
assert sorted(PIT_SYMBOLS) == symbols, (PIT_SYMBOLS, symbols)

pit_audit = {
    'snapshot_source': 'https://coinmarketcap.com/historical/20210101/',
    'snapshot_date': '2021-01-01',
    'verification_date': '2026-08-18',
    'verification_method': 'ccxt.binance().load_markets() — 실시간 조회',
    'cmc_top20_ex_stable_wrapped': [
        'BTC', 'ETH', 'XRP', 'LTC', 'DOT', 'BCH', 'ADA', 'BNB', 'LINK', 'BSV',
        'XLM', 'EOS', 'XMR', 'XEM', 'THETA', 'TRX', 'XTZ',
    ],  # USDT·USDC(스테이블코인), WBTC(래핑) 제외
    'included_13': PIT_SYMBOLS,
    'excluded_confirmed_inactive_4': {
        'BSVUSDT':  'active=False (ccxt 확인, 2026-08-18)',
        'EOSUSDT':  'active=False (ccxt 확인, 2026-08-18)',
        'XMRUSDT':  'active=False (ccxt 확인, 2026-08-18)',
        'XEMUSDT':  'active=False (ccxt 확인, 2026-08-18)',
    },
    'scope_limitation': (
        'top-20까지만 CMC 히스토리컬 스냅샷을 확인함(21~50위는 JS 렌더링 페이지네이션 때문에 '
        '스크래핑 불가) — 이 유니버스는 "2021년 1월 시총 top-20 중 바이낸스에서 지금도 '
        '거래 가능한 것"으로 엄격히 한정되며, 44종목 원 트랙과 같은 폭을 주장하지 않는다.'
    ),
}
with open(CRYPTO_DIR / 'crypto_pit_universe_audit.json', 'w', encoding='utf-8') as f:
    json.dump(pit_audit, f, ensure_ascii=False, indent=2)
print(json.dumps(pit_audit, ensure_ascii=False, indent=2))

## §C. 후속 강건성 진단 — 재학습 없이 이미 학습된 4β 풀 재사용

§B의 결과(검증 2024 강세장 → 테스트 2025-2026 급락, 격차 1.73)가 유난히 가혹한 한 번의 국면 전환이었는지, 이 방법론 자체의 반복되는 패턴인지 §B에서는 구분할 수 없었다. 아래는 **재학습 없이** 이미 학습된(2021-2023) 4개 β 고정 정책만 재사용해 두 가지를 확인한다: (C-2) 검증/테스트 경계를 여러 지점으로 밀어가며 반복 평가, (C-3) 4개 β가 실제로 서로 다른 정책이었는지(v3식 다양성 붕괴 재발 여부).

### C-1. 로더 (세션 재시작 대비)

In [ ]:
# 세션이 끊겨 pool_models가 메모리에 없어도 저장된 4β 풀만 다시 불러오면
# 재학습 없이 아래 진단을 이어서 실행할 수 있다.
if 'pool_models' not in globals() or not pool_models:
    pool_models = {b: PPO.load(str(POOL_DIR / f'agent_beta_{b}.zip')) for b in POOL_BETAS}
    print('4β 풀 로드 완료:', list(pool_models.keys()))
else:
    print('pool_models 이미 메모리에 있음 — 재사용')

assert 'env_kwargs' in globals(), 'env_kwargs 없음 — §B의 데이터 로드 셀(1~4)을 먼저 실행할 것'
assert 'run_standalone_smoothed' in globals(), '헬퍼 함수 없음 — §B-6 셀을 먼저 실행할 것'

### C-2. 롤링 검증/테스트 재분할 (재학습 없음, 6개월 블록 4개 fold)

§B-9의 1개 관측치(valid=2024 전체, test=2025-2026 전체, 격차 1.73)를 6개월 단위로 잘게 쪼갠 4개의 독립적인 (valid,test) fold로 확장한다. 각 fold는 여전히 "valid에서만 선택 → test 1회 평가" 원칙을 지킨다.

In [ ]:
# 2024-01-01 ~ 2026-06-30를 6개월 블록 5개로 나누고(2026-07~08은 데이터가 2개월뿐이라 제외),
# 인접한 블록 쌍마다 (valid=B_i, test=B_{i+1})로 §B-8/B-9와 동일한 절차를 반복한다:
#   1) 4β × 7α = 28개 조합을 valid 블록에서만 스윕해 검증 샤프 최고 (β,α) 선택
#   2) 그 (β,α)를 test 블록에서 1회 평가
# 재학습은 전혀 없다 — pool_models는 2021-2023으로 이미 고정 학습된 4개 정책.

BLOCK_BOUNDARIES = [
    ('2024-01-01', '2024-06-30'),
    ('2024-07-01', '2024-12-31'),
    ('2025-01-01', '2025-06-30'),
    ('2025-07-01', '2025-12-31'),
    ('2026-01-01', '2026-06-30'),
]

def block_buckets(start, end):
    return [b for b in all_buckets if start <= bucket_to_date(b) <= end]

blocks = [block_buckets(s, e) for s, e in BLOCK_BOUNDARIES]
for (s, e), blk in zip(BLOCK_BOUNDARIES, blocks):
    print(f'{s} ~ {e}: {len(blk):,}스텝' + ('' if blk else '  (데이터 없음 — 스킵됨)'))

rolling_rows = []
for i in range(len(blocks) - 1):
    v_buckets, t_buckets = blocks[i], blocks[i + 1]
    if not v_buckets or not t_buckets:
        continue
    v_label = f'{BLOCK_BOUNDARIES[i][0]}~{BLOCK_BOUNDARIES[i][1]}'
    t_label = f'{BLOCK_BOUNDARIES[i+1][0]}~{BLOCK_BOUNDARIES[i+1][1]}'

    # 1) 이 fold의 valid 블록에서 28개 조합 스윕
    fold_sweep = {}
    for beta in POOL_BETAS:
        model = pool_models[beta]
        for alpha in SMOOTH_ALPHAS:
            net, _, _ = run_standalone_smoothed(model, v_buckets, alpha)
            fold_sweep[(beta, alpha)] = compute_perf(net)['Sharpe']
    best_beta, best_alpha = max(fold_sweep, key=fold_sweep.get)

    # 2) 그 (β,α)로 이 fold의 test 블록을 1회 평가
    v_net, _, _ = run_standalone_smoothed(pool_models[best_beta], v_buckets, best_alpha)
    t_net, _, _ = run_standalone_smoothed(pool_models[best_beta], t_buckets, best_alpha)
    v_perf, t_perf = compute_perf(v_net), compute_perf(t_net)

    # 참조용 등가중
    ew_v, _ = equal_weight_backtest(CryptoPortfolioEnv(v_buckets, **env_kwargs))
    ew_t, _ = equal_weight_backtest(CryptoPortfolioEnv(t_buckets, **env_kwargs))

    rolling_rows.append({
        'valid_block': v_label, 'test_block': t_label,
        'selected_beta': best_beta, 'selected_alpha': best_alpha,
        'valid_sharpe': v_perf['Sharpe'], 'test_sharpe': t_perf['Sharpe'],
        'gap': abs(v_perf['Sharpe'] - t_perf['Sharpe']),
        'valid_ret': v_perf['총 수익률'], 'test_ret': t_perf['총 수익률'],
        'ew_valid_sharpe': compute_perf(ew_v)['Sharpe'], 'ew_test_sharpe': compute_perf(ew_t)['Sharpe'],
    })
    print(f'[{v_label} → {t_label}] 선택 β={best_beta},α={best_alpha} | '
          f'valid Sharpe {v_perf["Sharpe"]:.3f} → test Sharpe {t_perf["Sharpe"]:.3f} '
          f'(격차 {abs(v_perf["Sharpe"]-t_perf["Sharpe"]):.3f})')

rolling_df = pd.DataFrame(rolling_rows)
print()
print(rolling_df.to_string(index=False))
print()
print(f'평균 격차: {rolling_df["gap"].mean():.3f} | 중앙값 격차: {rolling_df["gap"].median():.3f} | '
      f'격차<0.3인 fold 수: {(rolling_df["gap"] < 0.3).sum()}/{len(rolling_df)}')
rolling_df.to_csv(CRYPTO_DIR / 'crypto_pit_rolling_folds.csv', index=False, encoding='utf-8-sig')
print('저장 완료: crypto_pit_rolling_folds.csv')

### C-3. 다양성 진단 — 4β가 실제로 서로 다른 정책이었나 (v3 재발 여부 확인)

In [ ]:
# v3(§4.3)에서 확인된 패턴 — Q-teacher 없이 우선순위 샘플링만으로는 45차원(여기선 13차원)
# 연속 행동공간에서 다양성이 만들어지지 않고 4개 β가 사실상 클론으로 수렴할 수 있음 —
# 이게 이 축소 유니버스에서도 재발했는지 비중 궤적 페어와이즈 상관으로 확인한다.
# 평가 구간은 테스트 블록 전체(2025-01~2026-08) — 실패가 관측된 바로 그 구간에서
# 정책들이 실제로 서로 달랐는지를 본다.

def collect_weight_trajectory(model, buckets):
    env = CryptoPortfolioEnv(buckets, **env_kwargs)
    obs, _ = env.reset()
    weights = [env.weights.copy()]
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(action)
        weights.append(env.weights.copy())
    return np.array(weights)  # (T, n_assets)

traj = {beta: collect_weight_trajectory(pool_models[beta], test_buckets) for beta in POOL_BETAS}

corr_rows = []
for i, b1 in enumerate(POOL_BETAS):
    for b2 in POOL_BETAS[i + 1:]:
        flat1, flat2 = traj[b1].ravel(), traj[b2].ravel()
        corr = float(np.corrcoef(flat1, flat2)[0, 1])
        corr_rows.append({'beta_pair': f'{b1} vs {b2}', 'weight_trajectory_corr': corr})

corr_df = pd.DataFrame(corr_rows)
print('4β 비중 궤적 페어와이즈 상관 (테스트구간 2025-2026, 1.0=완전 동일 정책)')
print(corr_df.to_string(index=False))
print()
print(f'평균 상관: {corr_df["weight_trajectory_corr"].mean():.3f}')
print('참고 — v3(44종목, §4.3): 사실상 전면 붕괴(거의 클론)')
print('참고 — v4(44종목, §4.4, 구조 개선 후): ρ 표준편차 0.068, β별 평균 ρ 0.109~0.545로 분리')
print('참고 — KOSPI 축소판(§7): 0.22~0.51(β=30만 구별) / 극단쌍 0.955(거의 동일)')

top10_by_beta = {
    beta: set(
        pd.Series(traj[beta].mean(axis=0)).nlargest(min(5, traj[beta].shape[1])).index
    )
    for beta in POOL_BETAS
}
print()
print('β별 평균비중 상위 5종목(13종목 중 — 44종목 top-10과 스케일 다르므로 축소 비교):')
# 주의: 비중 벡터의 인덱스 순서는 PIT_SYMBOLS(§A 정의 순서)가 아니라 §B에서
# h5 임베딩 파일 기준으로 재정렬된 `symbols`(알파벳순)를 따른다 — 반드시 이걸로 매핑.
for beta, top in top10_by_beta.items():
    names = [symbols[idx] for idx in top]
    print(f'  β={beta}: {names}')

corr_df.to_csv(CRYPTO_DIR / 'crypto_pit_diversity_diag.csv', index=False, encoding='utf-8-sig')
print('\n저장 완료: crypto_pit_diversity_diag.csv')

### C-4. 해석 가이드 (실행 후 이 기준으로 판단)

**C-2(롤링 재분할) 결과 읽는 법**:
- 4개 fold의 격차가 전부 크다(예: 대부분 >0.5) → §B의 실패가 2024→2025 특정 조합의 우연이 아니라 이 방법론(Q-teacher 없는 β-풀+valid 선택)의 일반적 취약성이라는 근거 강화.
- 일부 fold는 격차가 작다(<0.3) → 어떤 국면 전환은 견디고 어떤 전환(특히 강세→급락 같은 극단적 반전)은 못 견딘다는, 더 세밀한 조건부 결론으로 좁혀짐 — 이 경우 어떤 fold가 실패했는지(전환 방향·낙폭 등)를 §7 새 절에 구체적으로 적어야 함.

**C-3(다양성 진단) 결과 읽는 법**:
- 페어와이즈 상관이 v3처럼 사실상 1에 가깝다 → 애초에 4개가 클론이었으니 §B의 "β 선택"은 사실상 무의미한 절차였다는 뜻 — 실패 원인이 "walk-forward가 어려워서"가 아니라 "이번에도 다양성이 안 생겨서"로 재해석돼야 함(§4.3 원인의 재확인).
- 상관이 v4/KOSPI 수준으로 분리돼 있다(예: 평균 <0.7, β별로 다른 상위종목) → 4개는 진짜 서로 다른 정책이었는데도 전부 무너진 것 — 이러면 "다양성 부재" 가설로는 이번 실패를 설명할 수 없고, 순수하게 규제-없는 다년 일반화 실패로 봐야 함(더 근본적인 문제).

두 결과를 종합해 §7 새 절과 8장 한계#5의 정확한 문구를 다시 정하는 게 다음 단계.